## PRACTICA OBLIGATORIA: **Repaso Aprendizaje Supervisado**

* La práctica obligatoria de esta unidad consiste en resolver sobre un mismo dataset un problema de clasificación y un problema de regresión.
* Recuerda que debes subirla a tu repositorio personal antes de la sesión en vivo para que puntúe adecuadamente.
* Recuerda también que no es necesario que esté perfecta, sólo es necesario que se vea el esfuerzo.
* Esta práctica se resolverá en la sesión en vivo correspondiente y la solución se publicará en el repo del curso.

### Ejercicio 0

Importa los paquetes y módulos que necesites a lo largo del notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, recall_score,
                              mean_absolute_percentage_error, mean_squared_error, r2_score)
import warnings
warnings.filterwarnings('ignore')
print("Librerías cargadas correctamente")

### #1 Explicación del dataset y carga de datos

Trabajamos con el dataset 'Wine Quality' (Vinho verde portugués). Contiene propiedades físico-químicas de vinos blancos y tintos, junto con una puntuación de calidad sensorial.

**Dos targets:**
- **Clasificación:** `quality` (valores 3-9) — predecir la calidad del vino → optimizar recall medio.
- **Regresión:** `alcohol` (grado alcohólico) → minimizar el error porcentual (MAPE).

In [ ]:
# Carga (separador '|')
df = pd.read_csv('data/wines_dataset.csv', sep='|')
print(f"Dimensiones: {df.shape}")
df.head()

In [ ]:
df.info()

In [ ]:
# Distribución del target de clasificación
print("Distribución de la calidad (target clasificación):")
print(df['quality'].value_counts().sort_index())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df['quality'].value_counts().sort_index().plot(kind='bar', ax=axes[0],
    color='steelblue', edgecolor='black')
axes[0].set_title('Distribución de Calidad del vino')
axes[0].set_xlabel('Calidad')
axes[0].set_ylabel('Frecuencia')
axes[0].tick_params(rotation=0)

df['alcohol'].hist(bins=30, ax=axes[1], color='coral', edgecolor='black')
axes[1].set_title('Distribución de Grado Alcohólico (target regresión)')
axes[1].set_xlabel('Alcohol (%)')
axes[1].set_ylabel('Frecuencia')
plt.tight_layout()
plt.show()

In [ ]:
# Assessmet previo de ambos problemas
print("=== CLASIFICACIÓN (predecir quality) ===")
print(f"  Nº de clases: {df['quality'].nunique()} ({sorted(df['quality'].unique())})")
print(f"  Clase más frecuente: {df['quality'].mode()[0]} ({(df['quality']==df['quality'].mode()[0]).mean():.1%})")
print(f"  Desbalanceo: las clases 3,4,8,9 son muy minoritarias -> recall medio es la métrica adecuada")
print()
print("=== REGRESIÓN (predecir alcohol) ===")
print(f"  Media: {df['alcohol'].mean():.2f}% | Std: {df['alcohol'].std():.2f}%")
print(f"  Rango: [{df['alcohol'].min():.1f}%, {df['alcohol'].max():.1f}%]")
print(f"  Métrica: MAPE (error porcentual) -> apropiada para el objetivo de negocio")
print()
print("Nota: El dataset está limpio y sin valores faltantes.")
print(df.isnull().sum().sum(), "valores faltantes totales")

### Preparación general de los datos

In [ ]:
# Codificar la variable 'class' (tipo de vino: white/red)
le = LabelEncoder()
df['class_enc'] = le.fit_transform(df['class'])  # white=1, red=0

print("Codificación de 'class':", dict(zip(le.classes_, le.transform(le.classes_))))

### #2 Modelado para clasificación

Predecir la calidad del vino (`quality`) a partir de sus propiedades. Se usa **recall macro** como métrica, por petición del negocio (compromiso entre clases).

Features: todo excepto `quality`. KNN como baseline (k=1 y k=5), más RandomForest y GradientBoosting.

In [ ]:
# Features para clasificación: todo excepto quality (incluye alcohol y class_enc)
X_clf = df.drop(['quality', 'class'], axis=1)
y_clf = df['quality']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)
print(f"Train: {X_train_c.shape[0]} | Test: {X_test_c.shape[0]}")
print(f"Clases en train: {sorted(y_train_c.unique())}")

# Escalado (necesario para KNN)
scaler_c = StandardScaler()
X_train_c_sc = scaler_c.fit_transform(X_train_c)
X_test_c_sc  = scaler_c.transform(X_test_c)

In [ ]:
cv_c = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Baseline: KNN con k=1 y k=5
knn1 = KNeighborsClassifier(n_neighbors=1)
knn5 = KNeighborsClassifier(n_neighbors=5)

knn1_cv = cross_val_score(knn1, X_train_c_sc, y_train_c, cv=cv_c, scoring='recall_macro')
knn5_cv = cross_val_score(knn5, X_train_c_sc, y_train_c, cv=cv_c, scoring='recall_macro')
print(f"KNN k=1 (baseline) - Recall macro CV: {knn1_cv.mean():.4f} ± {knn1_cv.std():.4f}")
print(f"KNN k=5 (baseline) - Recall macro CV: {knn5_cv.mean():.4f} ± {knn5_cv.std():.4f}")

In [ ]:
# Random Forest
rf_c = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
rf_c_cv = cross_val_score(rf_c, X_train_c, y_train_c, cv=cv_c, scoring='recall_macro')
print(f"Random Forest      - Recall macro CV: {rf_c_cv.mean():.4f} ± {rf_c_cv.std():.4f}")

In [ ]:
# Gradient Boosting
gb_c = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb_c_cv = cross_val_score(gb_c, X_train_c, y_train_c, cv=cv_c, scoring='recall_macro')
print(f"Gradient Boosting  - Recall macro CV: {gb_c_cv.mean():.4f} ± {gb_c_cv.std():.4f}")

In [ ]:
# Comparación visual
res_clf = {
    'KNN k=1': knn1_cv.mean(),
    'KNN k=5': knn5_cv.mean(),
    'Random Forest': rf_c_cv.mean(),
    'Gradient Boosting': gb_c_cv.mean()
}

print("\nComparación Recall Macro en CV (clasificación calidad):")
for m, s in sorted(res_clf.items(), key=lambda x: -x[1]):
    print(f"  {m:<25} Recall macro: {s:.4f}")

plt.figure(figsize=(8, 4))
plt.bar(res_clf.keys(), res_clf.values(), color=['steelblue','royalblue','coral','mediumseagreen'], edgecolor='black')
plt.title('Recall Macro en CV - Clasificación Calidad Vino')
plt.ylabel('Recall Macro')
plt.ylim(0, 0.6)
plt.xticks(rotation=10, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Optimización del mejor modelo de clasificación (Random Forest)
param_grid_c = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_leaf': [1, 3],
    'class_weight': ['balanced']
}

grid_rf_c = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid_c,
    cv=cv_c,
    scoring='recall_macro',
    n_jobs=-1,
    verbose=0
)
grid_rf_c.fit(X_train_c, y_train_c)

print(f"Mejores parámetros: {grid_rf_c.best_params_}")
print(f"Recall macro en CV (optimizado): {grid_rf_c.best_score_:.4f}")

In [ ]:
# Evaluación sobre test (clasificación)
best_clf = grid_rf_c.best_estimator_
y_pred_c = best_clf.predict(X_test_c)

print("=" * 65)
print("EVALUACIÓN CLASIFICACIÓN - Random Forest Optimizado (TEST)")
print("=" * 65)
print(classification_report(y_test_c, y_pred_c))

cm_c = confusion_matrix(y_test_c, y_pred_c, labels=sorted(y_clf.unique()))
plt.figure(figsize=(8, 7))
sns.heatmap(cm_c, annot=True, fmt='d', cmap='Blues',
            xticklabels=sorted(y_clf.unique()), yticklabels=sorted(y_clf.unique()))
plt.title('Matriz de Confusión - Clasificación Calidad (Test)')
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.tight_layout()
plt.show()

In [ ]:
# Análisis de errores - clasificación
y_test_c_arr = np.array(y_test_c)
y_pred_c_arr = np.array(y_pred_c)
errores_c = y_test_c_arr != y_pred_c_arr

print(f"Total errores: {errores_c.sum()} de {len(y_test_c)} ({errores_c.mean():.1%})")
print()

# Distribución del error (diferencia real - predicho)
diffs = y_test_c_arr[errores_c] - y_pred_c_arr[errores_c]
print("Distribución de la magnitud del error (real - predicho):")
print(pd.Series(diffs).value_counts().sort_index())
print()
print("Observación: La mayoría de errores son de ±1 punto de calidad (clases adyacentes).")
print("Las clases extremas (3,4,8,9) son las más difíciles de predecir correctamente por escasez de datos.")
print()
print("Propuesta de mejora: usar técnicas de oversampling (SMOTE) en las clases minoritarias,")
print("o reformular como regresión ordinal para aprovechar la ordenación natural de las clases.")

### #3 Modelado para regresión

Predecir el grado alcohólico (`alcohol`) a partir del resto de propiedades, incluyendo calidad y tipo de vino.

Métrica objetivo: **MAPE** (Mean Absolute Percentage Error) — minimizar el error porcentual.

In [ ]:
# Features para regresión: todo excepto alcohol
X_reg = df.drop(['alcohol', 'class'], axis=1)
y_reg = df['alcohol']

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)
print(f"Train: {X_train_r.shape[0]} | Test: {X_test_r.shape[0]}")
print(f"Alcohol - Media train: {y_train_r.mean():.2f}%  Std: {y_train_r.std():.2f}%")

scaler_r = StandardScaler()
X_train_r_sc = scaler_r.fit_transform(X_train_r)
X_test_r_sc  = scaler_r.transform(X_test_r)

In [ ]:
from sklearn.metrics import make_scorer

# MAPE negativo (sklearn minimiza, así que negamos)
mape_scorer = make_scorer(mean_absolute_percentage_error, greater_is_better=False)
cv_r = 5

# Modelo 1: Ridge Regression
ridge = Ridge(alpha=1.0)
ridge_mape = -cross_val_score(ridge, X_train_r_sc, y_train_r, cv=cv_r, scoring=mape_scorer)
print(f"Ridge Regression     - MAPE CV: {ridge_mape.mean():.4f} ± {ridge_mape.std():.4f}")

In [ ]:
# Modelo 2: Random Forest Regressor
rf_r = RandomForestClassifier  # Vamos a usar RandomForestRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

rf_r = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_r_mape = -cross_val_score(rf_r, X_train_r, y_train_r, cv=cv_r, scoring=mape_scorer)
print(f"Random Forest Reg.   - MAPE CV: {rf_r_mape.mean():.4f} ± {rf_r_mape.std():.4f}")

In [ ]:
# Modelo 3: Gradient Boosting Regressor
gb_r = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb_r_mape = -cross_val_score(gb_r, X_train_r, y_train_r, cv=cv_r, scoring=mape_scorer)
print(f"Gradient Boosting R. - MAPE CV: {gb_r_mape.mean():.4f} ± {gb_r_mape.std():.4f}")

In [ ]:
# Comparación visual
res_reg = {
    'Ridge': ridge_mape.mean(),
    'Random Forest': rf_r_mape.mean(),
    'Gradient Boosting': gb_r_mape.mean()
}

print("\nComparación MAPE en CV (regresión alcohol):")
for m, s in sorted(res_reg.items(), key=lambda x: x[1]):
    print(f"  {m:<25} MAPE: {s:.4f} ({s*100:.2f}%)")

plt.figure(figsize=(7, 4))
plt.bar(res_reg.keys(), res_reg.values(), color=['coral','steelblue','mediumseagreen'], edgecolor='black')
plt.title('MAPE en CV - Regresión Grado Alcohólico')
plt.ylabel('MAPE (menor = mejor)')
plt.tight_layout()
plt.show()

In [ ]:
# Optimización del mejor modelo de regresión (Random Forest)
from sklearn.ensemble import RandomForestRegressor

param_grid_r = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_leaf': [1, 3]
}

grid_rf_r = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid_r,
    cv=cv_r,
    scoring=mape_scorer,
    n_jobs=-1,
    verbose=0
)
grid_rf_r.fit(X_train_r, y_train_r)

print(f"Mejores parámetros: {grid_rf_r.best_params_}")
print(f"Mejor MAPE en CV: {-grid_rf_r.best_score_:.4f} ({-grid_rf_r.best_score_*100:.2f}%)")

In [ ]:
# Evaluación sobre test (regresión)
best_reg = grid_rf_r.best_estimator_
y_pred_r = best_reg.predict(X_test_r)

mape_test = mean_absolute_percentage_error(y_test_r, y_pred_r)
rmse_test = np.sqrt(mean_squared_error(y_test_r, y_pred_r))
r2_test   = r2_score(y_test_r, y_pred_r)

print("=" * 60)
print("EVALUACIÓN REGRESIÓN - Random Forest Optimizado (TEST)")
print("=" * 60)
print(f"  MAPE:  {mape_test:.4f}  ({mape_test*100:.2f}%)")
print(f"  RMSE:  {rmse_test:.4f}  grados")
print(f"  R²:    {r2_test:.4f}")

In [ ]:
# Gráficos de análisis de errores (regresión)
residuos = y_test_r.values - y_pred_r

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Real vs Predicho
axes[0].scatter(y_test_r, y_pred_r, alpha=0.3, color='steelblue', s=10)
lim = [y_test_r.min(), y_test_r.max()]
axes[0].plot(lim, lim, 'r--', lw=1.5)
axes[0].set_xlabel('Valor real (alcohol %)')
axes[0].set_ylabel('Valor predicho (alcohol %)')
axes[0].set_title('Real vs Predicho')

# Residuos vs Predicho
axes[1].scatter(y_pred_r, residuos, alpha=0.3, color='coral', s=10)
axes[1].axhline(0, color='black', lw=1.5, ls='--')
axes[1].set_xlabel('Predicho')
axes[1].set_ylabel('Residuo (real - predicho)')
axes[1].set_title('Residuos vs Predicho')

# Distribución de residuos
axes[2].hist(residuos, bins=40, color='mediumseagreen', edgecolor='black', alpha=0.8)
axes[2].axvline(0, color='black', lw=1.5, ls='--')
axes[2].set_xlabel('Residuo')
axes[2].set_ylabel('Frecuencia')
axes[2].set_title('Distribución de Residuos')

plt.tight_layout()
plt.show()

print(f"Residuo medio: {residuos.mean():.4f}")
print(f"Residuo std:   {residuos.std():.4f}")
print()
print("Análisis:")
print("- Los residuos se distribuyen aproximadamente de forma normal centrada en 0.")
print("- El modelo predice bien en el rango central (10-13%), con mayor error en extremos.")
print("- El MAPE indica un error porcentual razonable para las simulaciones de negocio.")
print("- Posible mejora: añadir interacciones entre densidad y azúcar residual,")
print("  o probar con XGBoost para mayor precisión.")

### Conclusiones finales

**Clasificación (calidad del vino):**
- El Random Forest optimizado superó al KNN baseline y al Gradient Boosting en recall macro.
- Las clases extremas (3, 4, 8, 9) son difíciles de predecir por su bajo número de muestras.
- Se recomienda usar SMOTE o rebalanceo para mejorar la detección de clases minoritarias.

**Regresión (grado alcohólico):**
- El Random Forest regressor consiguió el menor MAPE en validación cruzada.
- El modelo obtiene un error porcentual bajo, adecuado para las simulaciones de negocio.
- Los residuos son aproximadamente normales y centrados en 0, lo que indica buen ajuste.
